# Проверка LTX Portrait Student — CelebV-HQ

Ноутбук выбирает веса из `outputs_ltx_student_celebv_hq` и использует точный deployment-режим обучения: 25 шагов и CFG 3.0.

- вход приводится к тому же face-centric 512×512 формату;
- conditioning: `portrait_prompt_embeds.pt`, T5 не загружается;
- VAE и scheduler: `Lightricks/LTX-Video`;
- transformer: студент 14×2048 (~983M);
- LTX генерирует 49 кадров (`8n+1`), экспорт добивает до 50 @ 25 FPS;
- при наличии `checkpoint-best/ema_transformer` генерируются online и EMA варианты;
- teacher reference создаётся с тем же seed и параметрами.

Сравнивайте `student_best.mp4`, `student_best_ema.mp4` и `teacher_reference.mp4`.

In [ ]:
import gc
import json
from datetime import datetime
from pathlib import Path

import torch
from diffusers import LTXImageToVideoPipeline, LTXVideoTransformer3DModel
from diffusers.utils import export_to_video
from PIL import Image

from celebv_hq_data import FaceCentricVideoConfig, read_portrait_conditioning_image
from preview_prompts import DEFAULT_NEGATIVE_PROMPT, DEFAULT_PREVIEW_PROMPT

PROJECT_DIR = Path.cwd()
WEIGHTS_ROOT = PROJECT_DIR / "outputs_ltx_student_celebv_hq"
IMAGE_PATH = PROJECT_DIR / "girl_project_crop_512x512.jpg"
EMBEDS_PATH = PROJECT_DIR / "portrait_prompt_embeds.pt"
TEACHER_MODEL_ID = "Lightricks/LTX-Video"
TEACHER_REVISION = "8984fa25007f376c1a299016d0957a37a2f797bb"

WIDTH = 512
HEIGHT = 512
NUM_FRAMES = 49
OUTPUT_FRAMES = 50
FPS = 25
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 3.0
DECODE_TIMESTEP = 0.0
DECODE_NOISE_SCALE = 0.0
SEED = 42
DTYPE = torch.bfloat16
GENERATE_EMA_CANDIDATE = True
GENERATE_TEACHER_REFERENCE = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("LTX preview requires CUDA")
print("device:", device)
print("project:", PROJECT_DIR)

In [ ]:
def is_transformer_dir(folder: Path) -> bool:
    return (folder / "config.json").exists() and bool(
        list(folder.glob("*.safetensors"))
    )


def find_preferred_student_dir(root: Path) -> Path:
    """Best validation checkpoint first; final/latest are fallbacks."""
    if not root.exists():
        raise FileNotFoundError(f"Нет каталога с весами: {root}")

    for checkpoint_name in ("checkpoint-best", "final", "checkpoint-latest"):
        folder = root / checkpoint_name / "student_transformer"
        if is_transformer_dir(folder):
            return folder

    candidates = []
    for folder in root.rglob("student_transformer"):
        if is_transformer_dir(folder):
            weights = list(folder.glob("*.safetensors"))
            candidates.append((max(path.stat().st_mtime for path in weights), folder))
    if not candidates:
        raise FileNotFoundError(f"В {root} нет student transformer")
    return max(candidates, key=lambda item: item[0])[1]


student_dir = find_preferred_student_dir(WEIGHTS_ROOT)
checkpoint_dir = student_dir.parent
checkpoint_name = checkpoint_dir.name
source_path = checkpoint_dir / "weights_source.json"
if source_path.exists():
    weights_source = json.loads(source_path.read_text(encoding="utf-8"))
else:
    weights_source = {
        "primary": "online" if checkpoint_name != "final" else "unknown",
        "ema_candidate": "ema_transformer" if checkpoint_name == "checkpoint-best" else None,
    }
primary_source = str(weights_source.get("primary", "unknown"))
ema_name = weights_source.get("ema_candidate")
ema_dir = checkpoint_dir / str(ema_name) if ema_name else None
if ema_dir is not None and (ema_dir == student_dir or not is_transformer_dir(ema_dir)):
    ema_dir = None
student_label = f"{primary_source} {checkpoint_name}"

weight_mtime = max(path.stat().st_mtime for path in student_dir.glob("*.safetensors"))
print("selected:", student_dir)
print("primary weights:", student_label)
print("EMA candidate:", ema_dir)
print("weights mtime:", datetime.fromtimestamp(weight_mtime).isoformat(sep=" ", timespec="seconds"))
metrics_path = checkpoint_dir / "validation_metrics.json"
if metrics_path.exists():
    print("validation metrics:", metrics_path.read_text(encoding="utf-8"))

In [ ]:
student = LTXVideoTransformer3DModel.from_pretrained(
    student_dir,
    torch_dtype=DTYPE,
    local_files_only=True,
).eval()

n_params = sum(parameter.numel() for parameter in student.parameters()) / 1e6
hidden_size = student.config.num_attention_heads * student.config.attention_head_dim
print(
    f"student params={n_params:.1f}M, layers={student.config.num_layers}, "
    f"hidden={hidden_size}"
)

pipe = LTXImageToVideoPipeline.from_pretrained(
    TEACHER_MODEL_ID,
    revision=TEACHER_REVISION,
    transformer=student,
    text_encoder=None,
    tokenizer=None,
    torch_dtype=DTYPE,
).to(device)
if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()
pipe.set_progress_bar_config(disable=False)

In [ ]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(IMAGE_PATH)
if not EMBEDS_PATH.exists():
    raise FileNotFoundError(
        f"{EMBEDS_PATH} is missing. Run: python save_portrait_embeds.py"
    )

input_cfg = FaceCentricVideoConfig(
    width=WIDTH,
    height=HEIGHT,
    frames=NUM_FRAMES,
    target_fps=FPS,
)
image_tensor, _ = read_portrait_conditioning_image(str(IMAGE_PATH), input_cfg)
image_array = (
    image_tensor.add(1.0)
    .mul(127.5)
    .clamp(0, 255)
    .byte()
    .permute(1, 2, 0)
    .numpy()
)
image = Image.fromarray(image_array, mode="RGB")
embeds = torch.load(EMBEDS_PATH, map_location="cpu", weights_only=False)
if (
    embeds.get("prompt") != DEFAULT_PREVIEW_PROMPT
    or embeds.get("negative_prompt") != DEFAULT_NEGATIVE_PROMPT
):
    raise RuntimeError(
        "portrait_prompt_embeds.pt was created for old preview prompts. "
        "Run: python save_portrait_embeds.py"
    )

prompt_embeds = embeds["prompt_embeds"].to(device=device, dtype=DTYPE)
prompt_attention_mask = embeds["prompt_attention_mask"].to(device=device)
negative_prompt_embeds = embeds["negative_prompt_embeds"].to(device=device, dtype=DTYPE)
negative_prompt_attention_mask = embeds["negative_prompt_attention_mask"].to(device=device)

print("image:", image.size)
print("prompt:", embeds.get("prompt"))
print("prompt_embeds:", tuple(prompt_embeds.shape), prompt_embeds.dtype)

In [ ]:
out_dir = PROJECT_DIR / "preview"
out_dir.mkdir(parents=True, exist_ok=True)


def generate_video(output_name: str):
    generator = torch.Generator(device=device).manual_seed(SEED)
    with torch.inference_mode():
        result = pipe(
            image=image,
            prompt=None,
            negative_prompt=None,
            prompt_embeds=prompt_embeds,
            prompt_attention_mask=prompt_attention_mask,
            negative_prompt_embeds=negative_prompt_embeds,
            negative_prompt_attention_mask=negative_prompt_attention_mask,
            width=WIDTH,
            height=HEIGHT,
            num_frames=NUM_FRAMES,
            frame_rate=FPS,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            decode_timestep=DECODE_TIMESTEP,
            decode_noise_scale=DECODE_NOISE_SCALE,
            generator=generator,
            output_type="pil",
        )
    frames = list(result.frames[0])
    while len(frames) < OUTPUT_FRAMES:
        frames.append(frames[-1])
    frames = frames[:OUTPUT_FRAMES]
    output_path = out_dir / output_name
    export_to_video(frames, str(output_path), fps=FPS)
    print(f"saved {len(frames)} frames -> {output_path}")
    return frames, output_path


student_frames, student_path = generate_video("student_best.mp4")
active_student = student
ema_frames = None
ema_path = None
teacher_path = None

if GENERATE_EMA_CANDIDATE and ema_dir is not None:
    active_student.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    ema_student = LTXVideoTransformer3DModel.from_pretrained(
        ema_dir,
        torch_dtype=DTYPE,
        local_files_only=True,
    ).to(device).eval()
    pipe.transformer = ema_student
    active_student = ema_student
    ema_frames, ema_path = generate_video("student_best_ema.mp4")

if GENERATE_TEACHER_REFERENCE:
    active_student.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    teacher = LTXVideoTransformer3DModel.from_pretrained(
        TEACHER_MODEL_ID,
        revision=TEACHER_REVISION,
        subfolder="transformer",
        torch_dtype=DTYPE,
    ).to(device).eval()
    pipe.transformer = teacher
    teacher_frames, teacher_path = generate_video("teacher_reference.mp4")
else:
    teacher_frames = None

In [ ]:
from IPython.display import Video, display

print("source")
display(image)
print(f"student ({student_label})")
display(Video(str(student_path), embed=True, width=512))
if ema_path is not None:
    print("student (EMA candidate)")
    display(Video(str(ema_path), embed=True, width=512))
if teacher_path is not None:
    print("teacher reference")
    display(Video(str(teacher_path), embed=True, width=512))